# Acquisizione laser + oscilloscopio (sessione unica)

Laser TLB-6700 e oscilloscopio nello stesso kernel Python, che resta vivo tra le celle (come il workspace MATLAB). Il laser si apre **una volta sola**, l'acquisizione (ultima cella) si può rilanciare più volte.

Il laser usa le stesse chiamate di `NP_USB_connect_velocity.m` e `trigger_scan.m` (indirizzo USB numerico, `OpenDevices(deviceID)`, `Query`/`Write`). Limiti di lunghezza d'onda e velocità di scan restano quelli impostati a mano sul laser: il codice non li scrive (scriverli prima dell'armamento dello scope impediva il trigger).

Ordine: celle 1, 2, 3, 4 una volta; poi la cella 5 per ogni acquisizione. Il notebook va aperto dalla sua cartella (`Chiara_Bru/runs_oscilloscope`).

In [ ]:
# Cella 1 - configurazione e funzioni dello scope
import os
import sys
import time
import datetime

import numpy as np
import pyvisa
import matplotlib.pyplot as plt

OSC_VISA = "USB::0x0699::0x0408::C028813::INSTR"
OSC_TIMEOUT_MS = 30000

CHANNELS = ["CH1", "CH2"]
LABELS = {"CH1": "CH1 (laser ramp)", "CH2": "CH2 (R spectrum)"}
COLORS = {"CH1": "tab:red", "CH2": "tab:blue"}

POLL_INTERVAL_S = 0.3
CHANNEL_READ_PAUSE_S = 0.5


def crea_cartella_run():
    oggi = datetime.date.today().strftime("%Y-%m-%d")
    base_folder = os.path.join(os.getcwd(), oggi)
    os.makedirs(base_folder, exist_ok=True)
    run_esistenti = [d for d in os.listdir(base_folder) if d.startswith("run_")]
    run_folder = os.path.join(base_folder, f"run_{len(run_esistenti) + 1:03d}")
    os.makedirs(run_folder)
    return run_folder


def configura_scope(osc):
    osc.write("HORizontal:MODE MANUAL")
    rec_len = int(osc.query("HORizontal:RECORDLENGTH?"))
    osc.write("DATA:START 1")
    osc.write(f"DATA:STOP {rec_len}")
    osc.write("DATA:WIDTH 1")
    osc.write("DATA:ENCdg ASCii")


def arm_acquisition(osc):
    osc.write("ACQ:STOPAFTER SEQUENCE")
    osc.write("ACQ:STATE RUN")
    # In ARMED lo scope sta acquisendo il pre-trigger (PrTrig) e ignora i
    # trigger: lo scan parte solo quando lo stato non è più ARMED (READY).
    print("Attendo che lo scope finisca il pre-trigger (ARMED -> READY)...")
    while osc.query("TRIGger:STATE?").strip().upper() == "ARMED":
        time.sleep(POLL_INTERVAL_S)
    print("Scope pronto ad accettare il trigger.")


def wait_for_acquisition(osc):
    print("Attendo che l'oscilloscopio completi l'acquisizione triggerata...")
    while True:
        if int(osc.query("ACQ:STATE?")) == 0:
            print("Acquisizione completata!")
            break
        time.sleep(POLL_INTERVAL_S)


def acquire_channel(osc, source="CH1"):
    osc.write(f"DATA:SOURCE {source}")
    ymult = float(osc.query("WFMPRE:YMULT?"))
    yoff = float(osc.query("WFMPRE:YOFF?"))
    yzero = float(osc.query("WFMPRE:YZERO?"))
    print(f"{source}: YMULT={ymult}, YOFF={yoff}, YZERO={yzero}")
    raw = osc.query("CURVE?")
    ydata = np.array(raw.split(","), dtype=float)
    return (ydata - yoff) * ymult + yzero

In [ ]:
# Cella 2 - connessione all'oscilloscopio (una volta per sessione)
rm = pyvisa.ResourceManager()
osc = rm.open_resource(OSC_VISA)
osc.timeout = OSC_TIMEOUT_MS
configura_scope(osc)
print("Oscilloscopio connesso e configurato.")

In [ ]:
# Cella 3 - laser: carica la DLL e apre il dispositivo (una volta per sessione)
USBADDR = 1
DEVICE_ID = 0x100A  # hex2dec('100A')

dll_path = os.path.abspath(os.path.join(os.getcwd(), "..", "..", "scripts", "laser", "UsbDllWrap.dll"))
if not os.path.isfile(dll_path):
    raise RuntimeError(f"UsbDllWrap.dll non trovata in {dll_path} (il notebook va aperto da Chiara_Bru/runs_oscilloscope)")
print(f"DLL trovata: {dll_path}")
sys.path.append(os.path.dirname(dll_path))

import clr
clr.AddReference("UsbDllWrap")
import Newport
from System.Text import StringBuilder

NP_USB = Newport.USBComm.USB()

esito = NP_USB.OpenDevices(DEVICE_ID)
print(f"OpenDevices({hex(DEVICE_ID)}) -> {esito}")
querydata = StringBuilder(64)


def laser_query(comando):
    esito = NP_USB.Query(USBADDR, comando, querydata)
    risposta = querydata.ToString()
    print(f"Query {comando!r} -> esito={esito}, risposta={risposta!r}")
    return risposta


def laser_write(comando):
    esito = NP_USB.Write(USBADDR, comando)
    print(f"Write {comando!r} -> esito={esito}")

In [ ]:
# Cella 4 - query di stato del laser (rilanciabile quando vuoi)
laser_query("*IDN?")
laser_query("OUTP:STAT?")

In [ ]:
# Cella 5 - UNA acquisizione sincronizzata (rilancia questa cella per ogni acquisizione)
run_folder = crea_cartella_run()
print(f"I dati saranno salvati in: {run_folder}")

arm_acquisition(osc)
laser_write("OUTP:SCAN:START")

wait_for_acquisition(osc)
time.sleep(1.0)

data = {}
for ch in CHANNELS:
    print(f"Leggo {ch}...")
    data[ch] = acquire_channel(osc, ch)
    time.sleep(CHANNEL_READ_PAUSE_S)

csv_path = os.path.join(run_folder, "oscilloscope_sweep.csv")
min_len = min(len(v) for v in data.values())
all_cols = np.column_stack([data[ch][:min_len] for ch in CHANNELS])
np.savetxt(csv_path, all_cols, delimiter=",", header=",".join(CHANNELS), comments="")
print(f"Dati salvati in: {csv_path}")

fig, axes = plt.subplots(len(CHANNELS), 1, figsize=(10, 6), sharex=True)
for ax, ch in zip(axes, CHANNELS):
    ax.plot(data[ch], color=COLORS[ch], lw=1.0, label=LABELS[ch])
    ax.set_ylabel("Voltage (V)", color=COLORS[ch])
    ax.tick_params(axis="y", labelcolor=COLORS[ch])
    ax.legend(loc="best")
axes[-1].set_xlabel("Sample index")
fig.suptitle(f"Acquisizione sincronizzata con scan del laser ({', '.join(CHANNELS)})")
plt.tight_layout()
plt.show()